# Go2: 速度変更・停止・左右旋回
固定した公式環境とPPOから、指令に追従する単一policyを学習します。
Colab T4で正式評価28/30成功、旧前進baselineは同条件0/30。平地の固定指令列での結果です。
停止/直進/その場旋回/歩行旋回を明示的に混合し、独立した30試行で評価します。
CPUでは学習しません。Colab GPUと既存Drive保存を利用します。


In [ ]:
# 01_baseline.ipynbで作成した自分のcheckpoint.ptを指定
BASELINE_CHECKPOINT = ""


## 1. 保存先・学習量

`USE_DRIVE=True`ならGoogle Driveの認証画面が出ます。専用フォルダにcheckpointと設定・ログを保存します。
FalseならColab切断で消えるため、最後にZIPをダウンロードしてください。
新しいセッションで再開する場合は同じセルから準備し直し、`RESUME_CHECKPOINT`へDrive上の**自分で作った**モデルの絶対パスを指定します。
モデルと同じフォルダのconfig.jsonが必要です。学習状態は復元しますが、乱数・物理状態を完全復元する再開ではありません。


In [ ]:
from pathlib import Path
import sys, os, json, subprocess, datetime, shutil, uuid, hashlib, urllib.request, difflib
assert (3, 10) <= sys.version_info[:2] < (3, 14), sys.version
assert sys.platform == "linux", "Use a Colab Linux GPU runtime"
BASE = Path("/content/genesis-go2")
BASE.mkdir(parents=True, exist_ok=True)
USE_DRIVE = True
NUM_ENVS = 1024       # OOMなら512、256へ。公式既定4096より小さく開始。
ITERATIONS = 301      # smokeとは別の本学習。学習不足ならcheckpointから追加。
SEED = 1
RESUME_CHECKPOINT = ""  # 例: /content/drive/MyDrive/genesis-go2/runs/.../model_100.pt
RECORD_VIDEO = True
BACKUP = None
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BACKUP = Path("/content/drive/MyDrive/genesis-go2/runs")
    BACKUP.mkdir(parents=True, exist_ok=True)
info = subprocess.check_output([
    "nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"], text=True)
print(info)
(BASE / "gpu.txt").write_text(info)
(BASE / "python.txt").write_text(sys.version)
print("Local:", BASE, "Backup:", BACKUP)


## 2. 独立したPython環境

Notebookの既存Torchは置換せず、専用venvを作ります。数GBのダウンロードと初回compileが発生します。
GPUメモリ表示だけでCUDA12.8互換は確定しないため、最後に実際のCUDA tensor演算を確認します。
ここで失敗したら下へ進まず、setup.logとgpu.txtを確認してください。
OSパッケージは描画用共有ライブラリのみで、ドライバやEGL設定ファイルの書換えは行いません。


In [ ]:
PY = BASE / "venv/bin/python"
setup_log = BASE / "setup.log"
def setup(command):
    print(" ".join(map(str, command)), flush=True)
    with setup_log.open("a") as log:
        p = subprocess.run(list(map(str, command)), stdout=log, stderr=subprocess.STDOUT)
    if p.returncode:
        print(setup_log.read_text()[-12000:])
        raise RuntimeError(f"Setup exit={p.returncode}; see {setup_log}")
if not PY.exists():
    # Colab's Python may omit ensurepip; uv seeds the venv without changing the notebook kernel.
    setup([sys.executable, "-m", "pip", "install", "--target", BASE / "bootstrap", "uv==0.9.2"])
    setup([BASE / "bootstrap/bin/uv", "venv", "--python", sys.executable,
           "--seed", "--cache-dir", BASE / "uv-cache", BASE / "venv"])
setup(["apt-get", "update", "-qq"])
setup(["apt-get", "install", "-y", "-qq", "libegl1", "libgl1", "libglib2.0-0"])
setup([PY, "-m", "pip", "install", "pip==25.3"])
setup([PY, "-m", "pip", "install", "torch==2.11.0+cu128",
       "--index-url", "https://download.pytorch.org/whl/cu128",
       "--report", str(BASE / "torch-install.json")])


In [ ]:
(BASE / "requirements.txt").write_text('genesis-world==1.4.5\nrsl-rl-lib==5.5.2\ntensorboard==2.20.0\n# torch==2.11.0+cu128 is installed separately from the official PyTorch index.\n# Transitive dependencies are resolved on Colab and recorded with pip freeze.\n')

setup([PY, "-m", "pip", "install", "-r", BASE / "requirements.txt",
       "--report", str(BASE / "genesis-install.json")])
setup([PY, "-m", "pip", "check"])
(BASE / "requirements.freeze.txt").write_text(subprocess.check_output(
    [str(PY), "-m", "pip", "freeze"], text=True))
setup([PY, "-c", "import torch; assert torch.cuda.is_available(); "
       "print(torch.__version__, torch.version.cuda, torch.cuda.get_device_name()); "
       "print((torch.ones(8, device='cuda') * 2).sum().item())"])
print("Setup OK. Freeze and pip download hashes recorded.")


## 3. 公式例の取得・録画用の最小差分

固定commitから2ファイルを取得しSHA-256を検証します。原ファイルは保持し、別コピーにcamera追加だけを適用します。
差分はcamera.patchへ保存します。reward・観測・制御は変更しません。
初期位置は[0,0,0.42]m、quaternionはwxyz=[1,0,0,0]。12関節はFR/FL/RR/RLのhip→thigh→calf順です。
角度rad、関節速度rad/s、制御50Hz、物理substep2、actionは基準角への0.25rad倍率の残差です。


In [ ]:
SPEC = {'repository': 'https://github.com/Genesis-Embodied-AI/Genesis', 'commit': '1ac23c39b32e567cc76e20a467fee99aaad257b7', 'engine_version': '1.4.5', 'files': {'go2_env.py': '22a08b9f1f5c765732a7b3d04c98b3cb6a25b565aa024479a0526bbc654c1a63', 'go2_train.py': '9d78b33a90e8833be648c302a818f1b10ebfb632d0f5de5214f68bf1d714cb00'}}

SOURCE = BASE / "source"
SOURCE.mkdir(exist_ok=True)
for name, expected in SPEC["files"].items():
    url = f'https://raw.githubusercontent.com/Genesis-Embodied-AI/Genesis/{SPEC["commit"]}/examples/locomotion/{name}'
    if (SOURCE / name).exists():
        data = (SOURCE / name).read_bytes()
    else:
        with urllib.request.urlopen(url, timeout=60) as response:
            data = response.read()
    assert hashlib.sha256(data).hexdigest() == expected, f"Source mismatch: {name}"
    (SOURCE / name).write_bytes(data)
(SOURCE / "upstream.lock.json").write_text(json.dumps(SPEC, indent=2))
original = (SOURCE / "go2_env.py").read_text()
anchor = "        self.scene.build(n_envs=num_envs)"
signature = "show_viewer=False):"
assert original.count(anchor) == original.count(signature) == 1
modified = original.replace(signature, "show_viewer=False, record=False):").replace(anchor,
    "        self.camera = None\n"
    "        if record:\n"
    "            self.camera = self.scene.add_camera(res=(640, 480), pos=(2, -2, 1.5), "
    "lookat=(0, 0, 0.3), GUI=False, env_idx=0)\n" + anchor)
(SOURCE / "go2_record_env.py").write_text(modified)
(SOURCE / "camera.patch").write_text("".join(difflib.unified_diff(
    original.splitlines(True), modified.splitlines(True), fromfile="go2_env.py", tofile="go2_record_env.py")))
print((SOURCE / "camera.patch").read_text())


## 指令実装の配置と検査
既存Colabではここから実行でき、依存導入を繰り返しません。


In [ ]:
from pathlib import Path
import os, sys, json, subprocess, datetime, uuid, shutil
BASE = Path("/content/genesis-go2")
PY = BASE / "venv/bin/python"
SOURCE = BASE / "source"
BACKUP = Path("/content/drive/MyDrive/genesis-go2")
assert PY.exists() and (SOURCE / "go2_record_env.py").exists(), "Run the setup/source cells first"
assert Path("/content/drive/MyDrive").is_dir(), "Mount your existing Drive first"
FILES = {'commands.py': '"""Command schedules and trajectory scoring; usable without Torch/Genesis."""\nimport json\nimport math\nfrom pathlib import Path\n\nCONFIG = json.loads(Path(__file__).with_name("command-config.json").read_text())\nMODES = ("stop", "straight", "turn", "arc")\n\n\ndef schedule(stage="full", mirror=1):\n    if mirror not in (-1, 1):\n        raise ValueError("mirror must be +1 or -1")\n    end = [3, 7, 11, 15, 19, 23, 27, 30]\n    commands = [(0, 0, 0), (0.2, 0, 0), (0.7, 0, 0), (0, 0, 0),\n                (0, 0, 0.5 * mirror), (0, 0, -0.5 * mirror),\n                (0.4, 0, 0.4 * mirror), (0, 0, 0)]\n    if stage == "speed":\n        commands = [(0, 0, 0), (0.2, 0, 0), (0.5, 0, 0), (0, 0, 0),\n                    (0.3, 0, 0), (0, 0, 0), (0.5, 0, 0), (0, 0, 0)]\n    return [{"start": 0 if i == 0 else end[i-1], "end": end[i], "command": list(c)}\n            for i, c in enumerate(commands)]\n\n\ndef command_at(step, stage="full", mirror=1):\n    # Integer boundaries avoid drift from repeatedly adding floating point dt.\n    for index, segment in enumerate(schedule(stage, mirror)):\n        if step < round(segment["end"] / CONFIG["dt"]):\n            return index, segment["command"]\n    raise ValueError("step outside schedule")\n\n\ndef cases(formal=False, stage="full", recording=False):\n    if recording:\n        return [{"seed": 90, "yaw_degrees": 0, "mirror": 1}]\n    seeds = CONFIG["formal_seeds"] if formal else [90]\n    yaws = CONFIG["formal_yaw_degrees"] if formal else [0]\n    mirrors = CONFIG["mirrors"] if stage == "full" else [1]\n    return [{"seed": seed, "yaw_degrees": yaw, "mirror": mirror}\n            for yaw in yaws for mirror in mirrors for seed in seeds]\n\n\ndef score_trial(rows, stage="full", mirror=1, thresholds=None):\n    limits = thresholds or CONFIG["thresholds"]\n    dt = CONFIG["dt"]\n    segments = []\n    failure = any(r["done"] for r in rows)\n    for index, seg in enumerate(schedule(stage, mirror)):\n        group = [r for r in rows if r["segment"] == index]\n        cmd = seg["command"]\n        stop = cmd == [0, 0, 0]\n        turn = cmd[0] == 0 and cmd[2] != 0\n        stable = [r for r in group if r["time"] - seg["start"] > limits["exclude_s"] and not r["done"]]\n        def mean(fn):\n            return sum(fn(r) for r in stable) / len(stable) if stable else None\n        vx = mean(lambda r: abs(r["vx"] - cmd[0]))\n        vy = mean(lambda r: abs(r["vy"]))\n        yaw = mean(lambda r: abs(r["wz"] - cmd[2]))\n        xy = mean(lambda r: math.hypot(r["vx"], r["vy"]))\n        consecutive, settled = 0, None\n        for row in group:\n            if stop:\n                good = math.hypot(row["vx"], row["vy"]) < limits["stop_xy_speed"] and abs(row["wz"]) < limits["stop_yaw_speed"]\n            else:\n                good = abs(row["vx"]-cmd[0]) <= limits["vx_mae"] and abs(row["vy"]) <= limits["vy_mae"] and abs(row["wz"]-cmd[2]) <= limits["yaw_mae"]\n                if turn:\n                    good = good and math.hypot(row["vx"], row["vy"]) <= limits["turn_xy_speed"]\n            consecutive = consecutive + 1 if good and not row["done"] else 0\n            if settled is None and consecutive >= round(limits["dwell_s"] / dt):\n                settled = row["time"] - seg["start"]\n        complete = len(group) == round((seg["end"]-seg["start"]) / dt)\n        tracked = vx is not None and vx <= limits["vx_mae"] and vy <= limits["vy_mae"] and yaw <= limits["yaw_mae"]\n        if turn:\n            tracked = tracked and xy <= limits["turn_xy_speed"]\n        distance = sum(math.hypot(r["vx"], r["vy"]) * dt for r in group)\n        segments.append({"index": index, **seg, "steps": len(group), "vx_mae": vx, "vy_mae": vy,\n                         "yaw_mae": yaw, "xy_speed_mean": xy, "settled_at_s": settled,\n                         "travel_distance_m": distance,\n                         "success": bool(complete and tracked and settled is not None and settled <= limits["settle_s"] and not any(r["done"] for r in group))})\n    return {"steps": len(rows), "terminated_early": failure,\n            "success": len(rows) == 1500 and not failure and all(s["success"] for s in segments),\n            "max_abs_roll_degrees": max((abs(r["roll"]) for r in rows), default=None),\n            "max_abs_pitch_degrees": max((abs(r["pitch"]) for r in rows), default=None),\n            "segments": segments}\n', 'command_env.py': '"""Extend the SHA-verified official environment without editing its source."""\nimport copy\nimport random\nimport torch\nimport genesis as gs\nfrom genesis.utils.geom import inv_quat, transform_by_quat, quat_to_xyz\nfrom go2_record_env import Go2Env\nfrom commands import CONFIG\n\n\nclass CommandEnv(Go2Env):\n    def __init__(self, num_envs, cfgs, stage="full", evaluation=False, record=False):\n        self.stage = copy.deepcopy(CONFIG["stages"][stage])\n        self.evaluation = evaluation\n        self._inside_step = False\n        self.terminal = None\n        super().__init__(num_envs, *copy.deepcopy(cfgs), record=record)\n        self.reward_values = {}\n        if evaluation:\n            # Capture the actual old-command rewards, before upstream resampling.\n            for name, function in list(self.reward_functions.items()):\n                def wrapped(fn=function, key=name):\n                    value = fn()\n                    self.reward_values[key] = (value * self.reward_scales[key]).detach().clone()\n                    return value\n                self.reward_functions[name] = wrapped\n\n    def _resample_commands(self, envs_idx):\n        if self.evaluation:\n            # Evaluation commands are injected explicitly, before observation/policy.\n            return\n        n = self.num_envs\n        modes = torch.multinomial(torch.tensor(self.stage["weights"], device=self.device), n, replacement=True)\n        vx = 0.1 + torch.rand(n, device=self.device) * (self.stage["vx_max"] - 0.1)\n        arc_vx = 0.1 + torch.rand(n, device=self.device) * (self.stage["arc_vx_max"] - 0.1)\n        yaw = (0.2 + torch.rand(n, device=self.device) * max(0.0, self.stage["yaw_max"] - 0.2))\n        yaw *= torch.where(torch.rand(n, device=self.device) < 0.5, -1.0, 1.0)\n        values = torch.zeros_like(self.commands)\n        values[:, 0] = torch.where(modes == 1, vx, torch.where(modes == 3, arc_vx, 0.0))\n        values[:, 2] = torch.where((modes == 2) | (modes == 3), yaw, 0.0)\n        if envs_idx is None:\n            self.commands.copy_(values)\n        else:\n            torch.where(envs_idx[:, None], values, self.commands, out=self.commands)\n\n    def set_commands(self, values):\n        values = torch.as_tensor(values, device=self.device, dtype=gs.tc_float)\n        if values.shape != self.commands.shape or not torch.isfinite(values).all():\n            raise ValueError("Invalid commands")\n        self.commands.copy_(values)\n        self._update_observation()\n        return self.get_observations()\n\n    def _reset_idx(self, envs_idx=None):\n        if self.evaluation and self._inside_step:\n            # Upstream updates these buffers, then calls this hook before resetting.\n            self.terminal = {k: getattr(self, k).detach().clone() for k in\n                             ["base_pos", "base_quat", "base_euler", "base_lin_vel", "base_ang_vel", "dof_vel", "reset_buf", "rew_buf"]}\n            self.terminal["solver_error"] = self.scene.rigid_solver.get_error_envs_mask().clone()\n        super()._reset_idx(envs_idx)\n        if self.evaluation:\n            if envs_idx is None:\n                self.commands.zero_()\n            else:\n                self.commands.masked_fill_(envs_idx[:, None], 0.0)\n\n    def step(self, actions):\n        self._inside_step = True\n        try:\n            return super().step(actions)\n        finally:\n            self._inside_step = False\n\n    def reset_with_yaws(self, yaw_degrees, seeds):\n        self.reset()\n        angles = torch.deg2rad(torch.tensor(yaw_degrees, device=self.device, dtype=gs.tc_float))\n        qpos = self.init_qpos.repeat(self.num_envs, 1)\n        qpos[:, 3] = torch.cos(angles / 2)\n        qpos[:, 4:6] = 0\n        qpos[:, 6] = torch.sin(angles / 2)\n        # Deterministic policy evaluation would otherwise repeat identical rollouts\n        # for every seed. Each seed defines a small, recorded initial joint offset.\n        offsets = []\n        for seed in seeds:\n            generator = random.Random(seed)\n            width = CONFIG["evaluation_joint_jitter_rad"]\n            offsets.append([generator.uniform(-width, width) for _ in range(qpos.shape[1] - 7)])\n        qpos[:, 7:] += torch.tensor(offsets, device=self.device, dtype=gs.tc_float)\n        self.evaluation_initial_qpos = qpos.detach().cpu().tolist()\n        self.robot.set_qpos(qpos, zero_velocity=True, skip_forward=False)\n        self.base_pos = self.robot.get_pos()\n        self.base_quat = self.robot.get_quat()\n        self.base_euler = quat_to_xyz(self.base_quat, rpy=True, degrees=True)\n        self.projected_gravity = transform_by_quat(self.global_gravity, inv_quat(self.base_quat))\n        self.dof_pos = self.robot.get_dofs_position(self.motors_dof_idx)\n        self.dof_vel.zero_()\n        self._update_observation()\n        return self.get_observations()\n', 'command_worker.py': '"""CUDA entry point for command-conditioned Go2 training, evaluation and teleoperation."""\nimport argparse\nimport copy\nimport csv\nimport hashlib\nimport importlib.metadata\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport shutil\nimport subprocess\nimport sys\nimport time\n\nfrom commands import CONFIG, cases, command_at, score_trial\n\n\ndef dump(path, value):\n    temp = path.with_suffix(path.suffix + ".partial")\n    temp.write_text(json.dumps(value, indent=2, allow_nan=False) + "\\n")\n    temp.replace(path)\n\n\ndef backup(run, destination):\n    if destination:\n        target = destination / run.name\n        target.mkdir(parents=True, exist_ok=True)\n        for path in run.iterdir():\n            if path.is_file() and not path.name.endswith(".partial"):\n                temp = target / (path.name + ".partial")\n                shutil.copy2(path, temp)\n                temp.replace(target / path.name)\n\n\ndef evaluate(env, runner, args, evaluation_stage):\n    import torch\n    import numpy as np\n    from PIL import Image, ImageDraw\n    trial_cases = cases(args.formal, evaluation_stage, args.mode in ["record", "teleop"])\n    obs = env.reset_with_yaws([c["yaw_degrees"] for c in trial_cases], [c["seed"] for c in trial_cases])\n    dump(args.run / "initial-states.json", [dict(c, qpos=q) for c, q in zip(trial_cases, env.evaluation_initial_qpos)])\n    policy = runner.get_inference_policy(device=env.device)\n    rows = [[] for _ in trial_cases]\n    active = [True] * len(trial_cases)\n    video_writer, video_stream = None, None\n    if args.mode in ["record", "teleop"]:\n        import av\n        video_writer = av.open(str(args.run / "commands.mp4"), "w")\n        video_stream = video_writer.add_stream("libx264", rate=25)\n        video_stream.width, video_stream.height, video_stream.pix_fmt = 640, 480, "yuv420p"\n    started = time.monotonic()\n    try:\n        with torch.no_grad():\n            for step in range(1500):\n                applied, segment_ids = [], []\n                for c in trial_cases:\n                    segment, cmd = command_at(step, evaluation_stage, c["mirror"])\n                    if args.mode == "teleop":\n                        requested = json.loads(args.control.read_text())\n                        cmd = [max(0, min(0.8, float(requested["vx"]))), 0,\n                               max(-0.6, min(0.6, float(requested["yaw"])))]\n                        if not all(math.isfinite(v) for v in cmd):\n                            raise ValueError("Non-finite control")\n                    applied.append(list(cmd))\n                    segment_ids.append(segment)\n                obs = env.set_commands(applied)\n                assert torch.allclose(obs["policy"][:, 6:9], env.commands * env.commands_scale)\n                actions = policy(obs)\n                if not torch.isfinite(actions).all():\n                    raise RuntimeError("Non-finite policy action")\n                obs, reward, _, _ = env.step(actions)\n                state = {k: v.detach().cpu().numpy() for k, v in env.terminal.items()}\n                if not all(np.isfinite(v).all() for v in state.values()) or not torch.isfinite(obs["policy"]).all():\n                    raise RuntimeError("Non-finite physics/observation")\n                action_values = actions.detach().cpu().tolist()\n                reward_terms = {k: v.cpu().tolist() for k, v in env.reward_values.items()}\n                for i, c in enumerate(trial_cases):\n                    if not active[i]:\n                        continue\n                    pos = state["base_pos"][i]\n                    quat = state["base_quat"][i]\n                    vel = state["base_lin_vel"][i]\n                    angular = state["base_ang_vel"][i]\n                    euler = state["base_euler"][i]\n                    done = bool(state["reset_buf"][i])\n                    row = {"step": step, "time": (step + 1) * 0.02, "segment": segment_ids[i],\n                           "cmd_vx": applied[i][0], "cmd_vy": applied[i][1], "cmd_yaw": applied[i][2],\n                           "vx": float(vel[0]), "vy": float(vel[1]), "vz": float(vel[2]),\n                           "wz": float(angular[2]), "x": float(pos[0]), "y": float(pos[1]), "z": float(pos[2]),\n                           "roll": float(euler[0]), "pitch": float(euler[1]),\n                           "yaw_world": math.atan2(2*(quat[0]*quat[3]+quat[1]*quat[2]), 1-2*(quat[2]**2+quat[3]**2)),\n                           "joint_speed_rms": float(np.sqrt(np.mean(state["dof_vel"][i] ** 2))),\n                           "reward": float(state["rew_buf"][i]), "done": done,\n                           "termination": "solver_error" if state["solver_error"][i] else ("roll_pitch_limit" if done else "none")}\n                    row.update({f"action_{j}": a for j, a in enumerate(action_values[i])})\n                    row.update({f"reward_{k}": v[i] for k, v in reward_terms.items()})\n                    rows[i].append(row)\n                    if done:\n                        active[i] = False\n                if video_writer and step % 2 == 0:\n                    row = rows[0][-1]\n                    if not row["done"]:\n                        pos = state["base_pos"][0]\n                        env.camera.set_pose(pos=pos + np.array([1.8, -2.0, 1.3]), lookat=pos)\n                        rgb = env.camera.render()[0]\n                        picture = Image.fromarray(rgb)\n                        draw = ImageDraw.Draw(picture)\n                        draw.rectangle((0, 0, 640, 56), fill="black")\n                        draw.text((8, 5), f\'t={row["time"]:.2f}s  command vx={row["cmd_vx"]:.2f}m/s yaw={row["cmd_yaw"]:.2f}rad/s\', fill="white")\n                        draw.text((8, 24), f\'actual vx={row["vx"]:.2f}m/s yaw={row["wz"]:.2f}rad/s  playback 1x\', fill="white")\n                        for packet in video_stream.encode(av.VideoFrame.from_ndarray(np.array(picture), format="rgb24")):\n                            video_writer.mux(packet)\n                        if args.mode == "teleop" and step % 10 == 0:\n                            picture.save(args.run / "latest.partial.jpg")\n                            (args.run / "latest.partial.jpg").replace(args.run / "latest.jpg")\n                if not any(active):\n                    break\n                if args.mode == "teleop":\n                    time.sleep(max(0, started + (step+1)*0.02 - time.monotonic()))\n    finally:\n        if video_writer:\n            for packet in video_stream.encode():\n                video_writer.mux(packet)\n            video_writer.close()\n        # Keep partial trajectories even if a renderer/physics failure interrupts the run.\n        with (args.run / "rollout.csv").open("w", newline="") as output:\n            nonempty = next((r for r in rows if r), None)\n            if nonempty:\n                writer = csv.DictWriter(output, fieldnames=["trial", "seed", "initial_yaw", "mirror", *nonempty[0]])\n                writer.writeheader()\n                for i, group in enumerate(rows):\n                    for r in group:\n                        writer.writerow({"trial": i, "seed": trial_cases[i]["seed"],\n                                         "initial_yaw": trial_cases[i]["yaw_degrees"], "mirror": trial_cases[i]["mirror"], **r})\n    trials = [{**c, **score_trial(r, evaluation_stage, c["mirror"])} for c, r in zip(trial_cases, rows)]\n    if args.mode == "teleop":\n        metrics = {"status": "manual_session_completed", "steps": len(rows[0]), "terminated_early": not active[0],\n                   "success_rate": None, "scope": "manual commands; not a formal evaluation"}\n    else:\n        metrics = {"status": "evaluated", "trials": trials,\n                   "success_rate": sum(t["success"] for t in trials) / len(trials),\n                   "formal": args.formal, "evaluation_stage": evaluation_stage,\n                   "scope": "flat ground; initial yaw and mirrored commands varied; no disturbance test"}\n        import matplotlib.pyplot as plt\n        example = rows[0]\n        fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)\n        for axis, measured, commanded, label in [(axes[0], "vx", "cmd_vx", "vx (m/s)"), (axes[1], "wz", "cmd_yaw", "yaw rate (rad/s)")]:\n            axis.plot([r["time"] for r in example], [r[commanded] for r in example], label="command")\n            axis.plot([r["time"] for r in example], [r[measured] for r in example], label="actual")\n            axis.set_ylabel(label)\n            axis.legend()\n        axes[1].set_xlabel("simulation time (s)")\n        fig.tight_layout()\n        fig.savefig(args.run / "tracking.png")\n        plt.close(fig)\n    return metrics\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("mode", choices=["smoke", "train", "eval", "record", "teleop"])\n    parser.add_argument("--run", required=True, type=Path)\n    parser.add_argument("--backup", type=Path)\n    parser.add_argument("--stage", choices=["speed", "full"])\n    parser.add_argument("--yaw-reward", type=float)\n    parser.add_argument("--num-envs", type=int, default=1024)\n    parser.add_argument("--iterations", type=int, default=301)\n    parser.add_argument("--seed", type=int, default=1)\n    parser.add_argument("--formal", action="store_true")\n    parser.add_argument("--control", type=Path)\n    group = parser.add_mutually_exclusive_group()\n    group.add_argument("--resume", type=Path)\n    group.add_argument("--initialize-from", type=Path)\n    group.add_argument("--checkpoint", type=Path)\n    args = parser.parse_args()\n    if args.num_envs < 1 or args.iterations < 1 or (args.mode == "teleop" and args.control is None):\n        parser.error("Invalid resource/control arguments")\n    if args.mode in ["eval", "record", "teleop"] and not args.checkpoint:\n        parser.error("Evaluation/recording requires --checkpoint")\n    args.run.mkdir(parents=True, exist_ok=False)\n    os.environ["MPLBACKEND"] = "Agg"\n    import torch\n    import genesis as gs\n    from command_env import CommandEnv\n    from go2_train import get_cfgs, get_train_cfg\n    from rsl_rl.runners import OnPolicyRunner\n    if not torch.cuda.is_available():\n        raise RuntimeError("CUDA required")\n    torch.set_num_threads(2)\n    spec = json.loads(Path(__file__).with_name("upstream.lock.json").read_text())\n    checkpoint = args.resume or args.initialize_from or args.checkpoint\n    saved = None\n    if checkpoint:\n        saved = json.loads((checkpoint.parent / "config.json").read_text())\n        if saved["upstream"] != spec:\n            raise ValueError("Upstream pins differ")\n    stage = args.stage or (saved.get("command_stage", "speed") if saved else "speed")\n    cfgs = get_cfgs()\n    train_cfg = get_train_cfg(args.run.name)\n    train_cfg["save_interval"] = 25\n    if args.resume:\n        if stage != saved["command_stage"] or saved["command_config"] != CONFIG:\n            raise ValueError("Resume must preserve stage and command configuration; use initialize-from for a new task")\n        cfgs, train_cfg = saved["cfgs"], saved["train_cfg"]\n        if args.yaw_reward is not None and args.yaw_reward != cfgs[2]["reward_scales"]["tracking_ang_vel"]:\n            raise ValueError("Resume cannot change the reward")\n    elif saved and args.checkpoint:\n        cfgs, train_cfg = saved["cfgs"], saved["train_cfg"]\n    elif args.yaw_reward is not None:\n        cfgs[2]["reward_scales"]["tracking_ang_vel"] = args.yaw_reward\n    if args.mode in ["eval", "record", "teleop"]:\n        cfgs = copy.deepcopy(cfgs)\n        cfgs[0]["episode_length_s"] = 35.0\n    cfgs[0]["resampling_time_s"] = CONFIG["hold_s"]\n    config = {"task": "go2-commands-v1", "command_stage": stage, "command_config": CONFIG,\n              "cfgs": cfgs, "train_cfg": train_cfg, "upstream": spec, "seed": args.seed,\n              "args": {k: str(v) if isinstance(v, Path) else v for k, v in vars(args).items()}}\n    dump(args.run / "config.json", config)\n    sources = {}\n    for name in ["command_worker.py", "commands.py", "command_env.py", "command-config.json", "go2_env.py", "go2_record_env.py", "go2_train.py", "upstream.lock.json"]:\n        source = Path(__file__).with_name(name)\n        shutil.copy2(source, args.run / name)\n        sources[name] = hashlib.sha256(source.read_bytes()).hexdigest()\n    dump(args.run / "source-sha256.json", sources)\n    dump(args.run / "dependencies.json", {p: importlib.metadata.version(p) for p in ["genesis-world", "torch", "rsl-rl-lib", "quadrants"]})\n    (args.run / "requirements.freeze.txt").write_text(subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True))\n    (args.run / "gpu.txt").write_text(subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"], text=True))\n    gs.init(backend=gs.cuda, precision="32", logging_level="warning", seed=args.seed, performance_mode=True)\n    is_eval = args.mode in ["eval", "record", "teleop"]\n    n = len(cases(args.formal, stage, args.mode in ["record", "teleop"])) if is_eval else (8 if args.mode == "smoke" else args.num_envs)\n    env = CommandEnv(n, cfgs, stage=stage, evaluation=is_eval, record=args.mode in ["record", "teleop"])\n    class SavedRunner(OnPolicyRunner):\n        def save(self, path, infos=None):\n            super().save(path, infos)\n            backup(args.run, args.backup)\n    runner = SavedRunner(env, copy.deepcopy(train_cfg), str(args.run), device=gs.device)\n    if checkpoint:\n        load_cfg = {"actor": True, "critic": True, "optimizer": False, "iteration": False} if args.initialize_from else None\n        runner.load(str(checkpoint), load_cfg=load_cfg, map_location=str(gs.device))\n        if args.resume:\n            runner.current_learning_iteration += 1\n    before_iteration = runner.current_learning_iteration\n    started = time.monotonic()\n    if is_eval:\n        metrics = evaluate(env, runner, args, stage)\n    else:\n        assert env.get_observations()["policy"].shape == (n, 45)\n        # Verify command injection and exact zero stop observations before training.\n        env.set_commands(torch.zeros(n, 3, device=gs.device))\n        assert torch.equal(env.get_observations()["policy"][:, 6:9], torch.zeros(n, 3, device=gs.device))\n        env.reset()\n        before = [p.detach().clone() for p in runner.alg.get_policy().parameters()]\n        iterations = 2 if args.mode == "smoke" else args.iterations\n        runner.learn(num_learning_iterations=iterations, init_at_random_ep_len=True)\n        delta = sum((a-b).abs().sum().item() for a, b in zip(runner.alg.get_policy().parameters(), before))\n        if not math.isfinite(delta) or delta <= 0:\n            raise RuntimeError("No finite PPO update")\n        runner.save(str(args.run / "checkpoint.pt"))\n        metrics = {"status": "ppo_updates_completed", "iterations_this_run": iterations,\n                   "transitions_this_run": n * train_cfg["num_steps_per_env"] * iterations,\n                   "start_iteration": before_iteration, "last_completed_iteration": runner.current_learning_iteration,\n                   "optimizer_restored": bool(args.resume), "parameter_l1_change": delta}\n    metrics["elapsed_s"] = time.monotonic() - started\n    dump(args.run / "metrics.json", metrics)\n    backup(args.run, args.backup)\n    print(json.dumps(metrics, indent=2), flush=True)\n\n\nif __name__ == "__main__":\n    try:\n        main()\n    except BaseException as exc:\n        # Parent orchestrator also persists an exit code when failure precedes imports.\n        print(f"COMMAND WORKER FAILED: {type(exc).__name__}: {exc}", flush=True)\n        raise\n', 'command_pipeline.py': '"""Bounded staged training pipeline; inspect status.json without duplicating jobs."""\nimport argparse\nfrom datetime import datetime, timezone\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport shutil\nimport subprocess\nimport sys\nimport time\nimport uuid\nimport zipfile\n\nfrom command_worker import backup, dump\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--session", type=Path, required=True)\n    parser.add_argument("--backup", type=Path, required=True)\n    parser.add_argument("--baseline", type=Path, required=True)\n    parser.add_argument("--num-envs", type=int, default=1024)\n    parser.add_argument("--iterations", type=int, default=301)\n    args = parser.parse_args()\n    args.session.mkdir(parents=True, exist_ok=False)\n    source = Path(__file__).parent\n    sequence = 0\n    history = []\n    chosen = None\n    def status(phase, **extra):\n        value = {"phase": phase, "updated_utc": datetime.now(timezone.utc).isoformat(),\n                 "history": history, **extra}\n        dump(args.session / "status.json", value)\n        args.backup.mkdir(parents=True, exist_ok=True)\n        shutil.copy2(args.session / "status.json", args.backup / "status.json")\n        print(json.dumps({k: v for k, v in value.items() if k != "history"}), flush=True)\n\n    def job(mode, *extra, timeout=1800):\n        nonlocal sequence\n        sequence += 1\n        run = args.session / "runs" / f"{sequence:02d}-{mode}-{uuid.uuid4().hex[:6]}"\n        log = args.session / "logs" / (run.name + ".log")\n        log.parent.mkdir(parents=True, exist_ok=True)\n        command = [sys.executable, "-u", str(source / "command_worker.py"), mode,\n                   "--run", str(run), "--backup", str(args.backup / "runs"), *map(str, extra)]\n        status("running", mode=mode, run=str(run), command=command)\n        code = -1\n        try:\n            with log.open("w") as output:\n                process = subprocess.Popen(command, cwd=source, stdout=output, stderr=subprocess.STDOUT,\n                                           env=dict(os.environ, MPLBACKEND="Agg", PYOPENGL_PLATFORM="egl",\n                                                    OMP_NUM_THREADS="2", OPENBLAS_NUM_THREADS="2"))\n                try:\n                    code = process.wait(timeout=timeout)\n                except BaseException:\n                    process.terminate()\n                    try:\n                        process.wait(timeout=15)\n                    except subprocess.TimeoutExpired:\n                        process.kill()\n                        process.wait()\n                    code = process.returncode\n                    raise\n        finally:\n            run.mkdir(parents=True, exist_ok=True)\n            shutil.copy2(log, run / "run.log")\n            dump(run / "command.json", command)\n            (run / "exit-code.txt").write_text(str(code))\n            backup(run, args.backup / "runs")\n        if code:\n            raise RuntimeError(f"{mode} failed exit={code}: {log}")\n        metrics = json.loads((run / "metrics.json").read_text())\n        history.append({"mode": mode, "run": str(run), "metrics": metrics})\n        status("job_completed", mode=mode, run=str(run), success_rate=metrics.get("success_rate"))\n        return run, metrics\n\n    try:\n        smoke, _ = job("smoke", "--stage", "full")\n        resumed, resumed_metrics = job("smoke", "--resume", smoke / "checkpoint.pt", "--stage", "full")\n        assert resumed_metrics["optimizer_restored"] and resumed_metrics["start_iteration"] == 2\n        status("resume_verified", run=str(resumed))\n        previous = None\n        for batch in range(3):\n            load = ["--resume", previous / "checkpoint.pt"] if previous else []\n            previous, _ = job("train", "--stage", "speed", "--num-envs", args.num_envs,\n                              "--iterations", args.iterations, *load)\n            _, metrics = job("eval", "--stage", "speed", "--checkpoint", previous / "checkpoint.pt")\n            if metrics["success_rate"] == 1.0:\n                break\n        # Keep the capability scope intact even if stage A needs more work: report its\n        # failures and give the full task a bounded attempt; never relabel it success.\n        full_previous = None\n        for batch in range(3):\n            load = ["--resume", full_previous / "checkpoint.pt"] if full_previous else ["--initialize-from", previous / "checkpoint.pt"]\n            full_previous, _ = job("train", "--stage", "full", "--num-envs", args.num_envs,\n                                   "--iterations", args.iterations, *load)\n            _, metrics = job("eval", "--stage", "full", "--checkpoint", full_previous / "checkpoint.pt")\n            if metrics["success_rate"] == 1.0:\n                break\n        if metrics["success_rate"] < 1.0:\n            # Plan\'s first reward adjustment: increase only yaw tracking weight.\n            status("yaw_reward_adjustment", old=0.2, new=0.5)\n            for batch in range(3):\n                load = (["--initialize-from", full_previous / "checkpoint.pt", "--yaw-reward", 0.5]\n                        if batch == 0 else ["--resume", full_previous / "checkpoint.pt"])\n                full_previous, _ = job("train", "--stage", "full", "--num-envs", args.num_envs,\n                                       "--iterations", args.iterations, *load)\n                _, metrics = job("eval", "--stage", "full", "--checkpoint", full_previous / "checkpoint.pt")\n                if metrics["success_rate"] == 1.0:\n                    break\n        chosen = full_previous / "checkpoint.pt"\n        formal_run, formal = job("eval", "--stage", "full", "--formal", "--checkpoint", chosen)\n        baseline_run, baseline = job("eval", "--stage", "full", "--formal", "--checkpoint", args.baseline)\n        recording, _ = job("record", "--stage", "full", "--checkpoint", chosen, timeout=1800)\n        summary = {"checkpoint": str(chosen), "checkpoint_drive": str(args.backup / "runs" / chosen.parent.name / chosen.name),\n                   "formal_run": str(formal_run), "formal_success_rate": formal["success_rate"],\n                   "baseline_success_rate": baseline["success_rate"], "recording": str(recording),\n                   "target_met": formal["success_rate"] >= 0.9,\n                   "scope": "speed, stop, yaw commands on flat ground; no disturbance or navigation"}\n        dump(args.session / "summary.json", summary)\n        shutil.copy2(args.session / "summary.json", args.backup / "summary.json")\n        archive = args.session / "go2-command-results.zip"\n        # Export final results and recipe, not every training checkpoint in every batch.\n        with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:\n            z.write(args.session / "summary.json", "summary.json")\n            for directory in [chosen.parent, formal_run, baseline_run, recording]:\n                for path in directory.iterdir():\n                    if path.is_file() and (not path.name.startswith("model_") or path.name == "checkpoint.pt"):\n                        z.write(path, str(path.relative_to(args.session)))\n        shutil.copy2(archive, args.backup / archive.name)\n        digest = hashlib.sha256(archive.read_bytes()).hexdigest()\n        if digest != hashlib.sha256((args.backup / archive.name).read_bytes()).hexdigest():\n            raise RuntimeError("Drive archive hash mismatch")\n        status("complete", **summary, archive=str(args.backup / archive.name), archive_sha256=digest)\n    except BaseException as exc:\n        status("failed", error=f"{type(exc).__name__}: {exc}", checkpoint=str(chosen) if chosen else None)\n        raise\n\n\nif __name__ == "__main__":\n    main()\n', 'command-config.json': '{\n  "schema": 1,\n  "dt": 0.02,\n  "hold_s": 4.0,\n  "evaluation_joint_jitter_rad": 0.01,\n  "stages": {\n    "speed": {"weights": [0.2, 0.8, 0.0, 0.0], "vx_max": 0.6, "arc_vx_max": 0.6, "yaw_max": 0.0},\n    "full": {"weights": [0.2, 0.3, 0.2, 0.3], "vx_max": 0.8, "arc_vx_max": 0.6, "yaw_max": 0.6}\n  },\n  "thresholds": {\n    "vx_mae": 0.10, "vy_mae": 0.10, "yaw_mae": 0.15,\n    "turn_xy_speed": 0.10, "stop_xy_speed": 0.05, "stop_yaw_speed": 0.10,\n    "settle_s": 2.0, "dwell_s": 0.5, "exclude_s": 1.0,\n    "formal_success_rate": 0.9\n  },\n  "formal_seeds": [1000, 1001, 1002, 1003, 1004],\n  "formal_yaw_degrees": [-15, 0, 15],\n  "mirrors": [1, -1]\n}\n', 'command_runtime_checks.py': '"""Check the actual CUDA command sampler without building a physics scene."""\nimport os\nos.environ["MPLBACKEND"] = "Agg"\nimport torch\nfrom command_env import CommandEnv\nfrom commands import CONFIG\n\ntorch.manual_seed(13)\nenv = object.__new__(CommandEnv)\nenv.num_envs = 10000\nenv.device = torch.device("cuda")\nenv.evaluation = False\nenv.stage = CONFIG["stages"]["full"]\nenv.commands = torch.zeros(env.num_envs, 3, device=env.device)\nenv._resample_commands(None)\nc = env.commands\nassert torch.all(c[:, 1] == 0)\nassert torch.all((c[:, 0] >= 0) & (c[:, 0] <= 0.8))\nassert torch.all(c[:, 2].abs() <= 0.6)\nstop = (c == 0).all(dim=1)\nturn = (c[:, 0] == 0) & (c[:, 2] != 0)\nassert 0.17 < stop.float().mean().item() < 0.23\nassert 0.17 < turn.float().mean().item() < 0.23\nassert (c[:, 2] > 0).any() and (c[:, 2] < 0).any()\nmask = torch.arange(env.num_envs, device=env.device) % 2 == 0\nbefore = c.clone()\nenv._resample_commands(mask)\nassert torch.equal(env.commands[~mask], before[~mask])\nenv.stage = CONFIG["stages"]["speed"]\nenv._resample_commands(None)\nassert torch.all(env.commands[:, 2] == 0)\nassert torch.all(env.commands[:, 0] <= 0.6)\nprint("CUDA sampler: exact stops, both yaw signs, bounds and selective reset PASS")\n', 'test_commands.py': '"""Meaningful, dependency-free tests for command boundaries and task scoring."""\nimport math\nimport unittest\nfrom commands import CONFIG, cases, command_at, score_trial\n\n\ndef ideal_rows(stage="full", mirror=1):\n    result = []\n    for step in range(1500):\n        segment, c = command_at(step, stage, mirror)\n        result.append({"step": step, "time": (step+1)*0.02, "segment": segment,\n                       "vx": c[0], "vy": 0.0, "wz": c[2], "done": False,\n                       "roll": 0.0, "pitch": 0.0})\n    return result\n\n\nclass CommandsTest(unittest.TestCase):\n    def test_switch_uses_next_observation_boundary(self):\n        self.assertEqual(command_at(149)[1], [0, 0, 0])\n        self.assertEqual(command_at(150)[1], [0.2, 0, 0])\n        self.assertEqual(command_at(750)[1], [0, 0, 0.5])\n        self.assertEqual(command_at(750, mirror=-1)[1], [0, 0, -0.5])\n        with self.assertRaises(ValueError):\n            command_at(1500)\n\n    def test_all_conditions_are_required(self):\n        self.assertTrue(score_trial(ideal_rows())["success"])\n        self.assertFalse(score_trial(ideal_rows()[:-1])["success"])\n        rows = ideal_rows()\n        rows[300]["done"] = True\n        self.assertFalse(score_trial(rows)["success"])\n\n    def test_standing_cannot_pass_motion_or_turns(self):\n        rows = ideal_rows()\n        for r in rows:\n            r.update(vx=0, vy=0, wz=0)\n        metrics = score_trial(rows)\n        self.assertFalse(metrics["success"])\n        self.assertTrue(metrics["segments"][0]["success"])\n        self.assertFalse(metrics["segments"][4]["success"])\n\n    def test_stop_requires_half_second_continuous_dwell(self):\n        rows = ideal_rows()\n        for r in rows:\n            if r["segment"] == 3:\n                r["vx"] = 0 if r["step"] % 24 < 23 else 0.1\n        self.assertIsNone(score_trial(rows)["segments"][3]["settled_at_s"])\n\n    def test_slow_stop_is_failure_even_if_late_mae_is_good(self):\n        rows = ideal_rows()\n        for r in rows:\n            if r["segment"] == 3 and r["time"] < 13.1:\n                r["vx"] = 0.06\n        seg = score_trial(rows)["segments"][3]\n        self.assertLess(seg["vx_mae"], 0.1)\n        self.assertFalse(seg["success"])\n\n    def test_formal_cases_include_both_turns_and_yaws(self):\n        c = cases(formal=True)\n        self.assertEqual(len(c), 30)\n        self.assertEqual({r["yaw_degrees"] for r in c}, {-15, 0, 15})\n        self.assertEqual({r["mirror"] for r in c}, {-1, 1})\n        self.assertTrue(set(CONFIG["formal_seeds"]).isdisjoint({r["seed"] for r in cases()}))\n\n\nif __name__ == "__main__":\n    unittest.main()\n', 'audit_commands.py': '"""Audit saved CSV/video/Drive copies without rerunning learning or physics."""\nimport csv\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\nimport shutil\nimport sys\n\n\ndef digest(path):\n    with path.open("rb") as stream:\n        return hashlib.file_digest(stream, "sha256").hexdigest()\n\n\ndef main(session):\n    state = json.loads((session / "status.json").read_text())\n    assert state["phase"] == "complete"\n    destination = Path(state["archive"]).parent\n    result = {"scope": "saved trajectory audit; no physics rerun", "evaluations": []}\n    # Independently spell out the intended schedule; do not reuse score_trial.\n    ends = [3, 7, 11, 15, 19, 23, 27, 30]\n    targets = [(0, 0), (.2, 0), (.7, 0), (0, 0), (0, .5), (0, -.5), (.4, .4), (0, 0)]\n    formal_jobs = [j for j in state["history"] if j["metrics"].get("formal")]\n    assert len(formal_jobs) == 2\n    initial_states = []\n    for job in formal_jobs:\n        run = Path(job["run"])\n        groups = {}\n        with (run / "rollout.csv").open() as stream:\n            for raw in csv.DictReader(stream):\n                row = {k: float(raw[k]) for k in ["time", "vx", "vy", "wz", "cmd_vx", "cmd_yaw"]}\n                row.update(step=int(raw["step"]), done=raw["done"] == "True")\n                assert all(math.isfinite(v) for v in row.values())\n                trial = int(raw["trial"])\n                groups.setdefault(trial, []).append(row)\n        assert len(groups) == 30\n        trials = []\n        for index, saved in enumerate(job["metrics"]["trials"]):\n            rows = groups[index]\n            assert [r["step"] for r in rows] == list(range(len(rows)))\n            segments = []\n            for segment, (vx_target, yaw_target) in enumerate(targets):\n                start = ends[segment-1] if segment else 0\n                lo, hi = round(start * 50), round(ends[segment] * 50)\n                block = rows[lo:hi]\n                yaw_target *= saved["mirror"]\n                for row in block:\n                    assert abs(row["time"] - (row["step"] + 1) / 50) < 1e-8\n                    assert abs(row["cmd_vx"] - vx_target) < 1e-8\n                    assert abs(row["cmd_yaw"] - yaw_target) < 1e-8\n                stable = [r for r in block if r["time"] - start > 1 and not r["done"]]\n                maes = [sum(abs(r[key] - target) for r in stable) / len(stable) if stable else None\n                        for key, target in [("vx", vx_target), ("vy", 0), ("wz", yaw_target)]]\n                streak, settled = 0, None\n                stop = vx_target == yaw_target == 0\n                turn = vx_target == 0 and yaw_target != 0\n                for r in block:\n                    xy = math.hypot(r["vx"], r["vy"])\n                    good = (xy < .05 and abs(r["wz"]) < .1) if stop else (\n                        abs(r["vx"]-vx_target) <= .1 and abs(r["vy"]) <= .1 and abs(r["wz"]-yaw_target) <= .15 and (not turn or xy <= .1))\n                    streak = streak + 1 if good and not r["done"] else 0\n                    if streak >= 25 and settled is None:\n                        settled = r["time"] - start\n                xy_mean = sum(math.hypot(r["vx"], r["vy"]) for r in stable) / len(stable) if stable else None\n                ok = bool(len(block) == hi-lo and stable and not any(r["done"] for r in block)\n                          and all(x <= bound for x, bound in zip(maes, [.1, .1, .15]))\n                          and (not turn or xy_mean <= .1) and settled is not None and settled <= 2)\n                assert ok == saved["segments"][segment]["success"]\n                segments.append({"index": segment, "success": ok, "mae_vx_vy_yaw": maes, "settled_at_s": settled})\n            success = len(rows) == 1500 and all(s["success"] for s in segments) and not any(r["done"] for r in rows)\n            assert success == saved["success"]\n            trials.append({"index": index, "seed": saved["seed"], "yaw_degrees": saved["yaw_degrees"],\n                           "mirror": saved["mirror"], "success": success,\n                           "failed_segments": [s for s in segments if not s["success"]]})\n        initial_states.append(json.loads((run / "initial-states.json").read_text()))\n        result["evaluations"].append({"run": run.name, "successes": sum(t["success"] for t in trials),\n                                      "trials": 30, "failures": [t for t in trials if not t["success"]]})\n    assert initial_states[0] == initial_states[1], "Baseline conditions differ"\n    import av\n    from PIL import Image, ImageDraw\n    recording = Path(state["recording"])\n    video = recording / "commands.mp4"\n    montage = Image.new("RGB", (960, 720))\n    selected = [25, 150, 250, 400, 500, 700]\n    with av.open(str(video)) as container:\n        stream = container.streams.video[0]\n        fps = float(stream.average_rate)\n        count = 0\n        for frame in container.decode(video=0):\n            if count in selected:\n                tile = frame.to_image().resize((480, 240))\n                ImageDraw.Draw(tile).text((5, 220), f"{count/fps:.1f}s", fill="yellow")\n                i = selected.index(count)\n                montage.paste(tile, ((i % 2)*480, (i//2)*240))\n            count += 1\n    assert count == 750 and fps == 25.0, (count, fps)\n    record_metrics = json.loads((recording / "metrics.json").read_text())\n    assert record_metrics["trials"][0]["steps"] == 1500\n    archive = session / Path(state["archive"]).name\n    assert digest(archive) == digest(Path(state["archive"])) == state["archive_sha256"]\n    assert digest(Path(state["checkpoint"])) == digest(Path(state["checkpoint_drive"]))\n    assert digest(video) == digest(destination / "runs" / recording.name / video.name)\n    result.update(video={"frames": count, "fps": fps, "seconds": count/fps, "sha256": digest(video)},\n                  archive_sha256=digest(archive), archive_bytes=archive.stat().st_size,\n                  checkpoint_sha256=digest(Path(state["checkpoint"])),\n                  recording_success=record_metrics["success_rate"], status="passed")\n    audit = session / "final-audit"\n    audit.mkdir(exist_ok=False)\n    (audit / "audit.json").write_text(json.dumps(result, indent=2) + "\\n")\n    shutil.copy2(__file__, audit / "audit_commands.py")\n    montage.save(audit / "preview.jpg")\n    shutil.copytree(audit, destination / "final-audit", dirs_exist_ok=False)\n    print(json.dumps(result, indent=2))\n\n\nif __name__ == "__main__":\n    main(Path(sys.argv[1]))\n'}
for name, text in FILES.items():
    (SOURCE / name).write_text(text)
runtime_env = dict(os.environ, MPLBACKEND="Agg", PYOPENGL_PLATFORM="egl", OMP_NUM_THREADS="2", OPENBLAS_NUM_THREADS="2")
for check in ["test_commands.py", "command_runtime_checks.py"]:
    checked = subprocess.run([str(PY), str(SOURCE / check)], env=runtime_env, capture_output=True, text=True)
    print(checked.stdout, checked.stderr)
    assert checked.returncode == 0, check
print("Command implementation deployed; unit and CUDA sampler checks passed")


## 一度だけ開始
短縮学習・中断再開probe→速度/停止→旋回→30試行評価→旧モデル比較→代表録画→Drive保存。
調整用seedと正式評価seedを分離。最大9バッチの学習に上限を設け、未達も保存します。


In [ ]:
# Starts background jobs once; use the status cell for subsequent checks.
import time
if "COMMAND_PROCESS" in globals() and COMMAND_PROCESS.poll() is None:
    raise RuntimeError("Command pipeline already running; do not launch a duplicate")
# Check here too: queued notebook cells can continue after an earlier cell fails.
for check in ["test_commands.py", "command_runtime_checks.py"]:
    subprocess.run([str(PY), str(SOURCE / check)], env=runtime_env, check=True)
BASELINE = Path(BASELINE_CHECKPOINT)
assert BASELINE.exists(), "Baseline checkpoint missing; choose your existing baseline path"
SESSION = BASE / "command-runs" / (datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:6])
DESTINATION = BACKUP / "command-runs" / SESSION.name
SESSION.parent.mkdir(parents=True, exist_ok=True)
pipeline_command = [str(PY), "-u", str(SOURCE / "command_pipeline.py"),
                    "--session", str(SESSION), "--backup", str(DESTINATION), "--baseline", str(BASELINE),
                    "--num-envs", str(globals().get("NUM_ENVS", 1024)),
                    "--iterations", str(globals().get("ITERATIONS", 301))]
pipeline_log = SESSION.parent / (SESSION.name + ".log")
with pipeline_log.open("w") as output:
    COMMAND_PROCESS = subprocess.Popen(pipeline_command, env=runtime_env, cwd=SOURCE,
                                       stdout=output, stderr=subprocess.STDOUT, start_new_session=True)
(SESSION.parent / (SESSION.name + ".launcher.json")).write_text(json.dumps({"pid": COMMAND_PROCESS.pid, "command": pipeline_command}, indent=2))
print("Started pipeline PID", COMMAND_PROCESS.pid)
print("SESSION", SESSION)
print("Drive", DESTINATION)
print("The cell has returned. Use the status cell; do not rerun setup or start.")


## 状態確認
このセルは何度実行しても学習を重複起動しません。GPU処理を開始したら定期チェックに戻ります。


In [ ]:
# Read-only: never starts/restarts a training job.
status_path = SESSION / "status.json"
if not status_path.exists():
    print("Starting. Log:", pipeline_log)
else:
    state = json.loads(status_path.read_text())
    print(json.dumps({k:v for k,v in state.items() if k != "history"}, indent=2))
    print("Completed jobs", len(state["history"]))
    if state["phase"] == "failed":
        print(pipeline_log.read_text()[-8000:])
    if state["phase"] == "running":
        log = SESSION / "logs" / (Path(state["run"]).name + ".log")
        if log.exists():
            print(log.read_text()[-2200:])
    if state["phase"] == "complete":
        from IPython.display import display, Image, Video
        for name in ["formal_run", "recording"]:
            graph = Path(state[name]) / "tracking.png"
            if graph.exists(): display(Image(filename=str(graph)))
        display(Video(str(Path(state["recording"]) / "commands.mp4"), embed=True))


## 保存結果の監査
全工程完了後に一度実行。CSVから独立再集計し、動画全フレームとDriveコピーのhashを確認します。物理・学習は再実行しません。


In [ ]:
assert json.loads((SESSION / "status.json").read_text())["phase"] == "complete"
checked = subprocess.run([str(PY), str(SOURCE / "audit_commands.py"), str(SESSION)],
                         env=runtime_env, capture_output=True, text=True)
print(checked.stdout, checked.stderr)
assert checked.returncode == 0


## 操作画面（任意）
評価完了後に実行。Stop motionはpolicyへ停止指令を送り、End sessionは処理を中断します。


In [ ]:
# Optional: after training, change vx/yaw sliders while watching the live image.
import ipywidgets as widgets
import threading, signal
from IPython.display import display
state = json.loads((SESSION / "status.json").read_text())
assert state["phase"] == "complete", "Wait for the pipeline first"
vx_slider = widgets.FloatSlider(min=0, max=0.8, step=0.05, description="vx m/s")
yaw_slider = widgets.FloatSlider(min=-0.6, max=0.6, step=0.05, description="yaw rad/s")
stop_button, start_button, end_button = widgets.Button(description="Stop motion"), widgets.Button(description="Start 30s session"), widgets.Button(description="End session")
live = widgets.Image(format="jpeg", width=640)
message = widgets.Label("Ready")
control_path = SESSION / "manual-control.json"
manual = {"process": None}
def update_control(change=None):
    temp = control_path.with_suffix(".partial")
    temp.write_text(json.dumps({"vx": vx_slider.value, "yaw": yaw_slider.value}))
    temp.replace(control_path)
def stop_motion(button):
    vx_slider.value, yaw_slider.value = 0.0, 0.0
    update_control()
def end_session(button):
    if manual["process"] and manual["process"].poll() is None:
        manual["process"].send_signal(signal.SIGINT)
def start_manual(button):
    if manual["process"] and manual["process"].poll() is None: return
    stop_motion(None)
    run = SESSION / "runs" / ("manual-" + uuid.uuid4().hex[:8])
    command = [str(PY), "-u", str(SOURCE / "command_worker.py"), "teleop", "--run", str(run),
               "--stage", "full", "--checkpoint", state["checkpoint"], "--control", str(control_path),
               "--backup", str(DESTINATION / "runs")]
    handle = (SESSION / "manual.log").open("w")
    process = subprocess.Popen(command, env=runtime_env, cwd=SOURCE, stdout=handle, stderr=subprocess.STDOUT)
    manual["process"] = process
    def watch():
        try:
            while process.poll() is None:
                image = run / "latest.jpg"
                if image.exists(): live.value = image.read_bytes()
                message.value = "Running; Stop motion sends zero speed; End session interrupts"
                time.sleep(0.2)
        finally:
            handle.close()
            message.value = f"Finished exit={process.returncode}; {run}"
            if run.exists(): shutil.copytree(run, DESTINATION / "runs" / run.name, dirs_exist_ok=True)
    threading.Thread(target=watch, daemon=True).start()
vx_slider.observe(update_control, names="value")
yaw_slider.observe(update_control, names="value")
stop_button.on_click(stop_motion)
start_button.on_click(start_manual)
end_button.on_click(end_session)
update_control()
display(widgets.VBox([vx_slider, yaw_slider, widgets.HBox([start_button, stop_button, end_button]), message, live]))
